In [1]:
from tqdm import tqdm
import numpy as np

import sympy as sp
from sympy import *

In [2]:
x = symbols('x', real=True)
lam = symbols('lambda', real=True, positive=True)

In [3]:
def raise_hyperbolic(n, psi):
    if n < 0:
        return psi
    result = -I * diff(psi, x) + I * (lam - n) * tanh(x) * psi
    return raise_hyperbolic(n - 1, result)

def get_aux_ground_state_hyperbolic(n):
    norm = sqrt(gamma(lam - n + Rational(1, 2)) / (sqrt(pi) * gamma(lam - n)))
    return norm * sech(x) ** (lam - n)

def get_eigenstate_hyperbolic(n):
    psi_0 = get_aux_ground_state_hyperbolic(n)
    norm = sqrt(gamma(2 * lam - 2*n + 1) / (gamma(n+1) * gamma(2 * lam - n + 1)))
    return (norm * raise_hyperbolic(n - 1, psi_0))

In [4]:
get_eigenstate_hyperbolic(0)

sqrt(gamma(lambda + 1/2))*sech(x)**lambda/(pi**(1/4)*sqrt(gamma(lambda)))

### Time Evolution

In [6]:
def change_basis_hyperbolic(psi_0, lam_val, precision=4):
    basis = []
    domain = np.linspace(-10, 10, 10**precision)
    psi_0 = lambdify(x, psi_0.subs(lam, lam_val), 'numpy')
    for n in tqdm(range(int(lam_val) + 1)):
        if n == lam_val:
            continue
        energy, psi_n = get_eigenstate_hyperbolic(n)
        energy = float(energy.evalf(subs={lam: lam_val}))
        psi_n = lambdify(x, conjugate(psi_n.subs(lam, lam_val)), 'numpy')
        coeff = np.trapz(psi_n(domain) * psi_0(domain), x=domain)
        basis.append((energy, coeff, psi_n))
    return basis


def time_evolve_hyperbolic(psi_transformed, max_t, frames=1000, tol=0.01):
    domain = np.linspace(-np.pi, np.pi, 1000)
    def psi_t(t_val):
        psi_t_vals = np.zeros_like(domain, dtype=complex)
        for energy, coeff, psi_n in psi_transformed:
            psi_t_vals += coeff * psi_n(domain) * np.exp(-1j * energy * t_val)
        return psi_t_vals
    
    psi_evolution = []
    for t in tqdm(np.linspace(0, max_t, frames)):
        psi_t_val = psi_t(t)
        if psi_evolution and abs(psi_t_val - psi_evolution[0][1]).max() < tol:
            print(f"Full period. Stopping evolution.")
            break
        psi_evolution.append((t, psi_t_val))
    return psi_evolution